# 텍스트 임베딩(TextEmbedding)

### 원핫인코딩(One-Hot Encodding)

In [1]:
from konlpy.tag import Okt  
okt = Okt()  
token = okt.morphs("나는 자연어 처리를 배운다")  
print(token)

word2index = {}
for voca in token:
    if voca not in word2index.keys():
        word2index[voca] = len(word2index)
print(word2index)

def one_hot_encoding(word, word2index):
    one_hot_vector = [0]*(len(word2index))
    index = word2index[word]
    one_hot_vector[index] = 1
    return one_hot_vector

one_hot_encoding("자연어",word2index)


['나', '는', '자연어', '처리', '를', '배운다']
{'나': 0, '는': 1, '자연어': 2, '처리': 3, '를': 4, '배운다': 5}


[0, 0, 1, 0, 0, 0]

위 코드의 결과를 보면, "자연어"라는 단어가 [0, 0, 1, 0, 0, 0]의 0과 1로 구성된 벡터로 표현되었다.

이처럼 표현하고 싶은 단어의 인덱스에 1의 값을 부여하고, 

다른 인덱스에는 0을 부여하는 단어의 벡터 표현 방식을 원-핫 벡터(One-Hot Vector)라고 한다.

### 희소 표현의 한계

위 원-핫 벡터처럼 벡터 또는 행렬(matrix)의 값이 대부분이 0으로 표현되는 방법을 

희소 표현(sparse representation)이라고 한다.

하지만 이런 희소 표현의 문제점은 단어의 개수가 늘어나면 벡터의 차원이 한없이 커진다는 점이다.

예를 들어 단어가 10000개 있고 "강아지"라는 단어의 인덱스가 5라면 [0, 0, 0, 0, 1, 0, 0, ...]이런식으로 표현이 된다.

이는 엄청난 공간적 낭비를 불러일으킨다.

또한 아래 예시를 살펴보면, 0과 1로만 구성된 원-핫 벡터로는 단어간 유사도를 구할 수 없다는 문제가 있다.

In [ ]:
import torch

# 원-핫 벡터 생성
dog = torch.FloatTensor([1, 0, 0, 0, 0])        # dog
cat = torch.FloatTensor([0, 1, 0, 0, 0])        # cat
computer = torch.FloatTensor([0, 0, 1, 0, 0])   # computer
netbook = torch.FloatTensor([0, 0, 0, 1, 0])    # netbook
book = torch.FloatTensor([0, 0, 0, 0, 1])       # book

print(torch.cosine_similarity(dog, cat, dim=0))
print(torch.cosine_similarity(cat, computer, dim=0))
print(torch.cosine_similarity(computer, netbook, dim=0))
print(torch.cosine_similarity(netbook, book, dim=0))

tensor(0.)
tensor(0.)
tensor(0.)
tensor(0.)


단어 간 유사도가 모두 0이 나온다.

### 밀집 표현(Dense Representation)과 워드 임베딩(Word Embedding)

밀집 표현은 희소 표현과 반대되는 개념이며, 

사용자가 설정한 값으로 모든 단어의 벡터 표현의 차원을 맞춘다.

또한, 이 과정에서 더 이상 0과 1만 가진 값이 아니라 실수값을 가지게 된다.

가령 "강아지" 단어를 4차원 벡터로 표현해본다면(예시로)

강아지 = [0.2, 1.8, 1.1 ,-2.1]

이런 식으로 표현할 수 있다.

이처럼 단어를 밀집 벡터(dense vector)의 형태로 표현하는 방법을 **워드 임베딩(word embedding)** 이라고 하며,

나온 결과를 **임베딩 벡터(embedding vector)** 라고 한다.

### Word2Vec

Word2Vec은 단어를 컴퓨터가 이해할 수 있는 수치적 벡터(Vector)로 변환하는 자연어 처리(NLP) 기술이다.

크게 2가지 구조로 나뉘는데, CBOW와 Skip-gram이다.

#### CBOW (Continuous Bag of Words)

주변에 있는 단어들을 보고 중심에 있는 단어가 무엇인지 맞히는 과정이다.

가령 예를 들어 "나른한 [ ? ] 앉아 졸고 있다" 라는 문장에서 빈 칸에 들어갈 고양이를 예측하는 과정 정도로 생각하면 된다.

I like studying data analysis.

라는 문장을 가지고 학습을 해본다고 하자.

먼저, 각 문장의 단어를 원-핫 벡터로 변환시킨다.

앞에 나온 순서대로 차례대로 만들어준다.

![CBOW1](imgs/cbow1.png)

중심 단어를 예측하기 위해서 앞, 뒤로 몇 개의 단어를 볼지를 결정하는데 이 범위를 윈도우(window)라고 한다.

예를 들어, studying이라는 단어를 예측하기 위해 window = 2로 설정하면

주변 단어는 앞뒤로 2단어, 즉 I, like, data, analysis가 된다.

즉, 이 말은 I, like, data, analysis를 이용해 studying이라는 단어를 예측한다는 말이 된다.

![cbow2](imgs/cbow2.png)

이제, 일반적인 신경망처럼 가중치 행렬을 곱해 주어야 한다.

이 가중치 행렬을 Lookup Table이라고 하며

사실상 학습이 끝난 후 이 가중치 테이블이 각 단어별 밀집 벡터라고 보면 된다.

여기서는 각 단어를 3차원의 벡터로 Embedding해보겠다.

아래 Lookup Table은 예시 랜덤값이다.

![cbow3](imgs/cbow3.png)

사실상 Lookup Table과 계산 결과가 거의 동일하게 나오는데(추출할 단어 studying 제외) 

이는 입력이 애초에 원-핫 벡터이기 때문이다.

또한 지금은 학습 중이라 확정을 할 순 없지만

학습이 끝난 후 Lookup Table이 저렇게 나왔다면

    I : [0.2, 0.7, 0.6]

    like : [0.3, 0.8, 0.7]

    studying : [0.5, 0.2, 0.4]

    data : [0.7, 0.7, 0.1]

    analysis : [0.9, 0.6, 0.4]

이런 식으로 밀집 벡터가 계산된 것이다.

이제 아래 그림처럼 연산 결과를 각 차원별로 평균내서 studying 단어의 벡터를 유추한다.

![cbow4](imgs/cbow4.png)

이제 studying에 대한 결과벡터를 내기 위한 연산과정에 들어간다.

마찬가지로 가중치 행렬을 곱해주고, 이후 Softmax함수를 거쳐 원핫벡터로 변환한다.

![cbow5](imgs/cbow5.png)

이후 역전파 과정을 거쳐 학습을 반복한다.

최종 정리하면 CBOW의 과정은 아래 그림과 같다.

![cbow6](imgs/cbow6.png)

CBOW 코드

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from collections import Counter

# =========================
# 1. 학습 문장 준비
# =========================
sentences = [
    "the fat cat sat on the mat",
    "the small cat sat on the sofa",
    "the dog sat on the mat",
    "the fat dog slept on the sofa",
    "the cat ate the fish",
    "the dog ate the bone",
    "a cat and a dog are animals",
    "the fish swam in the water",
    "the bird flew in the sky",
    "a dog ran in the park",
]

# 소문자 + 토큰화
tokenized = [s.lower().split() for s in sentences]

# =========================
# 2. 단어장 만들기
# =========================
words = []
for sentence in tokenized:
    words.extend(sentence)

vocab = sorted(set(words))
word_to_idx = {word: i for i, word in enumerate(vocab)}
idx_to_word = {i: word for word, i in word_to_idx.items()}

vocab_size = len(vocab)

print("단어장:")
print(word_to_idx)
print("vocab size:", vocab_size)

# =========================
# 3. CBOW 학습 데이터 만들기
# =========================
window_size = 2
training_data = []

for sentence in tokenized:
    for center_idx in range(window_size, len(sentence) - window_size):
        context_words = (
            sentence[center_idx - window_size:center_idx]
            + sentence[center_idx + 1:center_idx + window_size + 1]
        )
        target_word = sentence[center_idx]

        context_indices = [word_to_idx[w] for w in context_words]
        target_index = word_to_idx[target_word]

        training_data.append((context_indices, target_index))

print("\nCBOW 학습 데이터 예시:")
for context, target in training_data[:5]:
    print([idx_to_word[i] for i in context], "->", idx_to_word[target])

# =========================
# 4. CBOW 모델 정의
# =========================
class CBOW(nn.Module):
    def __init__(self, vocab_size, embedding_dim):
        super().__init__()

        # 이게 바로 lookup table W
        self.embeddings = nn.Embedding(vocab_size, embedding_dim)

        # 출력층 W'
        self.linear = nn.Linear(embedding_dim, vocab_size)

    def forward(self, context_indices):
        # context_indices shape: [context_size]

        # lookup table 통과
        embeds = self.embeddings(context_indices)
        # embeds shape: [context_size, embedding_dim]

        # CBOW 평균 벡터 h
        h = embeds.mean(dim=0)
        # h shape: [embedding_dim]

        # hW' 계산 → logits
        logits = self.linear(h)
        # logits shape: [vocab_size]

        return logits, embeds, h

# =========================
# 5. 모델 학습
# =========================
embedding_dim = 5
model = CBOW(vocab_size, embedding_dim)

loss_function = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=0.05)

epochs = 300

for epoch in range(epochs):
    total_loss = 0

    for context_indices, target_index in training_data:
        context_tensor = torch.tensor(context_indices, dtype=torch.long)
        target_tensor = torch.tensor([target_index], dtype=torch.long)

        optimizer.zero_grad()

        logits, embeds, h = model(context_tensor)

        # CrossEntropyLoss는 입력 shape이 [batch, class]여야 함
        loss = loss_function(logits.unsqueeze(0), target_tensor)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    if (epoch + 1) % 50 == 0:
        print(f"Epoch {epoch + 1}, Loss: {total_loss:.4f}")

# =========================
# 6. 최종 lookup table W 출력
# =========================
print("\n최종 Lookup Table W")
print(model.embeddings.weight.data)

# =========================
# 7. 단어별 밀집벡터 출력
# =========================
print("\n단어별 임베딩 벡터")
for word in vocab:
    idx = word_to_idx[word]
    vector = model.embeddings.weight.data[idx]
    print(f"{word:10s} -> {vector.numpy()}")

# =========================
# 8. 특정 입력이 lookup table을 통과하는 과정 보기
# =========================
example_context = ["the", "fat", "sat", "on"]
example_indices = [word_to_idx[w] for w in example_context]
example_tensor = torch.tensor(example_indices, dtype=torch.long)

logits, embeds, h = model(example_tensor)

print("\n예시 context:", example_context)
print("\nlookup table을 통과한 벡터들 H:")
print(embeds.data)

print("\nCBOW 평균 벡터 h:")
print(h.data)

print("\nhW' 결과 logits:")
print(logits.data)

단어장:
{'a': 0, 'and': 1, 'animals': 2, 'are': 3, 'ate': 4, 'bird': 5, 'bone': 6, 'cat': 7, 'dog': 8, 'fat': 9, 'fish': 10, 'flew': 11, 'in': 12, 'mat': 13, 'on': 14, 'park': 15, 'ran': 16, 'sat': 17, 'sky': 18, 'slept': 19, 'small': 20, 'sofa': 21, 'swam': 22, 'the': 23, 'water': 24}
vocab size: 25

CBOW 학습 데이터 예시:
['the', 'fat', 'sat', 'on'] -> cat
['fat', 'cat', 'on', 'the'] -> sat
['cat', 'sat', 'the', 'mat'] -> on
['the', 'small', 'sat', 'on'] -> cat
['small', 'cat', 'on', 'the'] -> sat
Epoch 50, Loss: 14.7794
Epoch 100, Loss: 4.9028
Epoch 150, Loss: 2.2951
Epoch 200, Loss: 1.3464
Epoch 250, Loss: 0.9082
Epoch 300, Loss: 0.6685

최종 Lookup Table W
tensor([[ 0.3260,  4.1202,  2.1039, -0.9217, -1.9934],
        [-1.1205, -0.3854,  0.0234,  0.4797, -2.8663],
        [ 1.1377,  0.9551, -1.2208,  0.7960, -0.9488],
        [ 0.2365, -0.1722, -1.2215, -0.0747, -0.4423],
        [ 0.0683, -0.5982,  2.0234,  0.3330,  1.1381],
        [ 0.6846, -0.5239, -0.8308,  0.8303, -2.3157],
        [-0.

#### Skip-gram

CBOW가 주변 단어를 통해 중심 단어를 예측하는 모델이라면, Skip-gram은 그 반대, 즉 중심 단어를 통해 주변 단어를 예측하는 모델이다.

"[ ? ] 고양이 [ ? ]" 이런 문장에서 나른한, 앉아 를 예측하는 정도로 생각하면 된다.

![Skipgram1](imgs/skipgram1.png)

W`의 파라미터들을 4번의 학습을 통해, Cross-entropy를 최소화 하는 방향으로 학습한다.

일반적으로 CBOW는 하나의 중심단어에 대해서 단 한 번 파라미터가 업데이트되지만, 

Skip-gram은 window size만큼 학습이 되므로, 

일반적으로 CBOW보다 Skip-gram이 더 좋은 성능을 보여주는 것으로 알려져있다.

Skip-gram 실습코드

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim

# =========================
# 1. 학습 문장 준비
# =========================
sentences = [
    "the fat cat sat on the mat",
    "the small cat sat on the sofa",
    "the dog sat on the mat",
    "the fat dog slept on the sofa",
    "the cat ate the fish",
    "the dog ate the bone",
    "a cat and a dog are animals",
    "the fish swam in the water",
    "the bird flew in the sky",
    "a dog ran in the park",
]

tokenized = [s.lower().split() for s in sentences]

# =========================
# 2. 단어장 생성
# =========================
words = []

for sentence in tokenized:
    words.extend(sentence)

vocab = sorted(set(words))

word_to_idx = {w: i for i, w in enumerate(vocab)}
idx_to_word = {i: w for w, i in word_to_idx.items()}

vocab_size = len(vocab)

print("Vocabulary")
print(word_to_idx)
print("vocab_size =", vocab_size)

# =========================
# 3. Skip-gram 학습 데이터 생성
# =========================
window_size = 2

training_data = []

for sentence in tokenized:

    for center_idx in range(len(sentence)):

        center_word = sentence[center_idx]

        # 주변 단어 범위
        start = max(0, center_idx - window_size)
        end = min(len(sentence), center_idx + window_size + 1)

        context_words = []

        for idx in range(start, end):

            if idx != center_idx:
                context_words.append(sentence[idx])

        # Skip-gram:
        # 중심 단어 하나로 주변 단어 각각 예측
        for context_word in context_words:

            center_index = word_to_idx[center_word]
            context_index = word_to_idx[context_word]

            training_data.append((center_index, context_index))

# 학습 데이터 예시 출력
print("\nSkip-gram Training Examples")

for i in range(20):
    center, context = training_data[i]

    print(
        idx_to_word[center],
        "->",
        idx_to_word[context]
    )

# =========================
# 4. Skip-gram 모델 정의
# =========================
class SkipGram(nn.Module):

    def __init__(self, vocab_size, embedding_dim):

        super().__init__()

        # Lookup Table W
        self.embeddings = nn.Embedding(
            vocab_size,
            embedding_dim
        )

        # 출력층 W'
        self.linear = nn.Linear(
            embedding_dim,
            vocab_size
        )

    def forward(self, center_word):

        # lookup table 통과
        embed = self.embeddings(center_word)
        # shape: [embedding_dim]

        # hW'
        logits = self.linear(embed)
        # shape: [vocab_size]

        return logits, embed

# =========================
# 5. 모델 생성
# =========================
embedding_dim = 5

model = SkipGram(
    vocab_size,
    embedding_dim
)

loss_function = nn.CrossEntropyLoss()

optimizer = optim.SGD(
    model.parameters(),
    lr=0.05
)

# =========================
# 6. 학습
# =========================
epochs = 300

for epoch in range(epochs):

    total_loss = 0

    for center_idx, context_idx in training_data:

        center_tensor = torch.tensor(
            center_idx,
            dtype=torch.long
        )

        context_tensor = torch.tensor(
            [context_idx],
            dtype=torch.long
        )

        optimizer.zero_grad()

        logits, embed = model(center_tensor)

        loss = loss_function(
            logits.unsqueeze(0),
            context_tensor
        )

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    if (epoch + 1) % 50 == 0:
        print(
            f"Epoch {epoch+1}, Loss: {total_loss:.4f}"
        )

# =========================
# 7. 최종 Lookup Table W
# =========================
print("\nFinal Lookup Table W")

print(model.embeddings.weight.data)

# =========================
# 8. 단어별 임베딩 벡터 출력
# =========================
print("\nWord Embeddings")

for word in vocab:

    idx = word_to_idx[word]

    vector = model.embeddings.weight.data[idx]

    print(
        f"{word:10s} -> {vector.numpy()}"
    )

# =========================
# 9. 특정 단어 lookup 과정 확인
# =========================
example_word = "cat"

example_idx = word_to_idx[example_word]

example_tensor = torch.tensor(
    example_idx,
    dtype=torch.long
)

logits, embed = model(example_tensor)

print("\n=========================")
print("Example Skip-gram")
print("=========================")

print("Center Word:")
print(example_word)

print("\nLookup Table Output (Embedding):")
print(embed.data)

print("\nhW' Result (logits):")
print(logits.data)

# =========================
# 10. softmax 확률 보기
# =========================
probabilities = torch.softmax(
    logits,
    dim=0
)

print("\nSoftmax Probabilities")

for idx, prob in enumerate(probabilities):

    print(
        f"{idx_to_word[idx]:10s}: {prob.item():.4f}"
    )

Vocabulary
{'a': 0, 'and': 1, 'animals': 2, 'are': 3, 'ate': 4, 'bird': 5, 'bone': 6, 'cat': 7, 'dog': 8, 'fat': 9, 'fish': 10, 'flew': 11, 'in': 12, 'mat': 13, 'on': 14, 'park': 15, 'ran': 16, 'sat': 17, 'sky': 18, 'slept': 19, 'small': 20, 'sofa': 21, 'swam': 22, 'the': 23, 'water': 24}
vocab_size = 25

Skip-gram Training Examples
the -> fat
the -> cat
fat -> the
fat -> cat
fat -> sat
cat -> the
cat -> fat
cat -> sat
cat -> on
sat -> fat
sat -> cat
sat -> on
sat -> the
on -> cat
on -> sat
on -> the
on -> mat
the -> sat
the -> on
the -> mat
Epoch 50, Loss: 415.6394
Epoch 100, Loss: 407.3965
Epoch 150, Loss: 405.7743
Epoch 200, Loss: 405.5063
Epoch 250, Loss: 405.6516
Epoch 300, Loss: 405.9547

Final Lookup Table W
tensor([[ 0.8314, -2.1460, -1.6221, -0.0228, -1.2590],
        [-0.1123,  0.9524, -3.0204,  0.4431, -0.3269],
        [ 1.5902, -1.5935, -3.1225,  0.2379,  0.0583],
        [ 0.8213, -0.3645, -2.5798,  1.8138,  1.4945],
        [ 0.2169,  2.1915, -0.2586,  0.8038, -1.3453],


### Word2vec의 한계

Word2Vec의 마지막 단계를 살펴보자. 

출력층에 있는 소프트맥스 함수는 단어 집합 크기의 벡터 내의 모든 값을 0과 1사이의 값이면서 모두 더하면 1이 되도록 바꾸는 작업을 수행한다. 

그리고 이에 대한 오차를 구하고 모든 단어에 대한 임베딩을 조정한다. 

그 단어가 중심 단어나 주변 단어와 전혀 상관없는 단어라도 말이다. 

지금은 단어 집합의 크기가 작아 상관없지만

만약 단어 집합의 크기가 수백만에 달한다면 이 작업은 굉장히 무거운 작업이 될 것이다.

결국 핵심은 모든 단어 집합에 대해서 연산을 수행한다는 것이다.

이를 해결하기 위해 Word2vec은 일반적으로 SGNS(Skip-Gram with Negative Sampling)을 사용한다.

Skip-gram 방식에 Negative Sampling을 추가로 수행한다는 것인데,

Negative Sampling에 대해 설명하자면

'돈가스', '컴퓨터', '회의실'과 같은 랜덤으로 선택된 주변 단어가 아닌 상관없는 단어들을 일부만 가져 온 후, 

이전체 단어 집합보다 훨씬 작은 단어 집합을 만들어놓고 마지막 단계를 이진 분류 문제로 바꿔버린다. 

즉, Word2Vec은 주변 단어들을 긍정(positive)으로 두고 

랜덤으로 샘플링 된 단어들을 부정(negative)으로 둔 다음에 이진 분류 문제를 수행한다.

"나는 어제 공원에서 귀여운 강아지와 산책을 했다"

라는 문장을 가지고 확인해보자.

#### 1. vocab만들기

|인덱스|단어|
|:---:|:---:|
|0|나는|
|1|어제|
|2|공원에서|
|3|귀여운|
|4|강아지와|
|5|산책을|
|6|했다|
|7|컴퓨터|
|8|돈가스|
|9|회의실|

#### 2. 입출력행렬 W, W'(초기 랜덤값)
가령 초기 입출력행렬 W, W'의 값이 아래와 같다고 해보자.

$$
W = \begin{bmatrix}
0.1 & 0.0 & 0.3 \\
0.2 & 0.1 & 0.0 \\
0.7 & 0.2 & 0.1 \\
0.6 & 0.4 & 0.3 \\
0.5 & 0.2 & 0.4 \\
0.3 & 0.7 & 0.6 \\
0.1 & 0.5 & 0.8 \\
0.9 & 0.1 & 0.2 \\
0.4 & 0.8 & 0.3 \\
0.2 & 0.9 & 0.7
\end{bmatrix}
\quad , \quad
W' = \begin{bmatrix}
0.3 & 0.6 & 0.1 \\
0.2 & 0.5 & 0.4 \\
0.8 & 0.1 & 0.3 \\
0.7 & 0.2 & 0.6 \\
0.4 & 0.9 & 0.5 \\
0.1 & 0.3 & 0.8 \\
0.5 & 0.4 & 0.2 \\
0.9 & 0.7 & 0.1 \\
0.6 & 0.8 & 0.4 \\
0.2 & 0.1 & 0.9
\end{bmatrix}
$$

#### 3. 중심 단어에 대한 연산 수행

예를 들어 중심단어 "강아지와"는 인덱스가 4이므로 입력 원핫 벡터는 $[0, 0, 0, 0, 1, 0, 0, 0, 0]$가 되고,

연산을 수행한 결과는 $v_c = [0.5, 0.2, 0.4]$가 된다.


#### 4-1. Positive Pair

window = 2라고 했을 때

주변단어는 "공원에서", "귀여운", "산책을", "했다" 가 된다.

"귀여운"을 예시로 계산해 보자면

먼저 "귀여운" 단어의 분류기 벡터를 W'에서 가져온다.

즉 $u_o = [0.7, 0.2, 0.6]$이 된다.

이제 중심 단어 벡터와 주변 단어 분류기 벡터의 내적 연산을 취한다.

계산하면

$[0.5, 0.2, 0.4] \cdot [0.7, 0.2, 0.6] = 0.63$, 

그리고 이 값을 Sigmoid화 해주면

$Sigmoid(0.63) \approx 0.652$

즉 "귀여운"이 중심단어 "강아지와"의 주변단어일 확률은 약 65.2%라는 의미이다.

이제 이 값이 1이 되도록 학습이 진행된다.

두 벡터 $v_c$, $u_o$의 내적이 커지는 방향으로 학습이 진행되어야 하기 때문에

가중치가 증가하는 방향(+)으로 학습을 진행시킨다.

$$
v_{c}^{new} = v_{c} + \eta \frac{\partial L}{\partial v_{c}} \quad , \quad u_{o}^{new} = u_{o} + \eta \frac{\partial L}{\partial u_{o}}
$$

여기서

$$
\frac{\partial L}{\partial v_{c}} = (y - Sigmoid(v_{c} \cdot u_{o}))u_{o} \quad , \quad \frac{\partial L}{\partial u_{o}} = (y - Sigmoid(v_{c} \cdot u_{o}))v_{c}
$$

즉 

$$
\frac{\partial L}{\partial v_{c}} = 0.3475 \times \begin{bmatrix} 0.7 \\ 0.2 \\ 0.6 \end{bmatrix} = \begin{bmatrix} 0.2433 \\ 0.0695 \\ 0.2085 \end{bmatrix}
$$

$$
\frac{\partial L}{\partial u_{o}} = 0.3475 \times \begin{bmatrix} 0.5 \\ 0.2 \\ 0.4 \end{bmatrix} = \begin{bmatrix} 0.1738 \\ 0.0695 \\ 0.1390 \end{bmatrix}
$$

이를 이용해 두 벡터를 갱신하면

$$
v_{c}^{new} = \begin{bmatrix} 0.5 \\ 0.2 \\ 0.4 \end{bmatrix} + 0.1 \times \begin{bmatrix} 0.2433 \\ 0.0695 \\ 0.2085 \end{bmatrix} = \begin{bmatrix} 0.5243 \\ 0.2070 \\ 0.4209 \end{bmatrix}
$$

$$
u_{o}^{new} = \begin{bmatrix} 0.7 \\ 0.2 \\ 0.6 \end{bmatrix} + 0.1 \times \begin{bmatrix} 0.1738 \\ 0.0695 \\ 0.1390 \end{bmatrix} = \begin{bmatrix} 0.7174 \\ 0.2070 \\ 0.6139 \end{bmatrix}
$$

이런 방식으로 학습한다.

#### 4-2. Negative Pair

주변단어가 아닌 임의의 단어 "컴퓨터"을 예시로 계산해 보자면

먼저 "컴퓨터" 단어의 분류기 벡터를 W'에서 가져온다.

즉 $u_o = [0.9, 0.7, 0.1]$이 된다.

이제 중심 단어 벡터와 주변 단어 분류기 벡터의 내적 연산을 취한다.

계산하면

$[0.5, 0.2, 0.4] \cdot [0.9, 0.7, 0.1] = 0.63$, 

그리고 이 값을 Sigmoid화 해주면

$Sigmoid(0.63) \approx 0.652$

즉 "컴퓨터"가 중심단어 "강아지와"의 주변단어일 확률은 약 65.2%라는 의미이다.

이제 이 값이 0이 되도록 학습이 진행된다.

$$
v_{c}^{new} = v_{c} + \eta \frac{\partial L}{\partial v_{c}} \quad , \quad u_{o}^{new} = u_{o} + \eta \frac{\partial L}{\partial u_{o}}
$$

여기서

$$
\frac{\partial L}{\partial v_{c}} = (y - Sigmoid(v_{c} \cdot u_{o}))u_{o} \quad , \quad \frac{\partial L}{\partial u_{o}} = (y - Sigmoid(v_{c} \cdot u_{o}))v_{c}
$$

즉 

$$
\frac{\partial L}{\partial v_{c}} = - 0.652 \times \begin{bmatrix} 0.9 \\ 0.7 \\ 0.1 \end{bmatrix} = - \begin{bmatrix} 0.5868 \\ 0.4564 \\ 0.0652 \end{bmatrix}
$$

$$
\frac{\partial L}{\partial u_{o}} = - 0.652 \times \begin{bmatrix} 0.5 \\ 0.2 \\ 0.4 \end{bmatrix} = - \begin{bmatrix} 0.326 \\ 0.1304 \\ 0.2608 \end{bmatrix}
$$

이를 이용해 두 벡터를 갱신하면

$$
v_{c}^{new} = \begin{bmatrix} 0.5 \\ 0.2 \\ 0.4 \end{bmatrix} - 0.1 \times \begin{bmatrix} 0.5868 \\ 0.4564 \\ 0.0652 \end{bmatrix} = \begin{bmatrix} 0.4413 \\ 0.1544 \\ 0.3935 \end{bmatrix}
$$

$$
u_{o}^{new} = \begin{bmatrix} 0.9 \\ 0.7 \\ 0.1 \end{bmatrix} - 0.1 \times \begin{bmatrix} 0.326 \\ 0.1304 \\ 0.2608 \end{bmatrix} = \begin{bmatrix} 0.8674 \\ 0.6870 \\ 0.0740 \end{bmatrix}
$$

이런 방식으로 학습한다.